# Notebook 02 — Final speaker split, 10-fold groups, and raw MFCC extraction

This version prepares the data for the final training run. It keeps the existing 42/11 speaker split, creates 10 speaker-grouped folds inside the 42-speaker training partition, and saves **raw MFCCs**. Normalization is deliberately performed separately inside each fold in Notebook 03.

The original recordings are identified using `is_augmented == False`. Augmented recordings are used for training, while validation and final-test metrics are calculated on original recordings only.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import joblib

from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder

SEED = 42
N_FOLDS = 10
OUT = Path("anu_revision_output")
OUT.mkdir(exist_ok=True)

manifest = pd.read_csv(OUT / "dataset_manifest_TO_FILL.csv").fillna("")
required = ["path", "label", "speaker_id", "original_file", "is_augmented"]
for col in required:
    assert col in manifest.columns, f"Missing column: {col}"
    assert manifest[col].astype(str).str.strip().ne("").all(), f"Blank values in {col}"

manifest["is_augmented"] = (
    manifest["is_augmented"].astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes"])
)
manifest["path"] = manifest["path"].astype(str)
manifest["speaker_id"] = manifest["speaker_id"].astype(str)
manifest["original_file"] = manifest["original_file"].astype(str)
manifest["label"] = manifest["label"].astype(str)

missing = [p for p in manifest["path"] if not Path(p).exists()]
assert not missing, f"{len(missing)} audio paths do not exist. Example: {missing[:3]}"

# One original recording and every derivative must map to exactly one speaker and one label.
assert (manifest.groupby("original_file")["speaker_id"].nunique() == 1).all()
assert (manifest.groupby("original_file")["label"].nunique() == 1).all()

# Every original recording must have exactly one non-augmented row.
original_counts = manifest.groupby("original_file")["is_augmented"].apply(
    lambda x: int((~x).sum())
)
assert (original_counts == 1).all(), "Each original_file must have exactly one original WAV."

print("Total WAV files:", len(manifest))
print("Original recordings:", int((~manifest["is_augmented"]).sum()))
print("Augmented recordings:", int(manifest["is_augmented"].sum()))
print("Classes:", manifest["label"].nunique())
print("Speaker IDs:", manifest["speaker_id"].nunique())

Total WAV files: 4440
Original recordings: 740
Augmented recordings: 3700
Classes: 14
Speaker IDs: 53


In [2]:
# Keep the same outer speaker split protocol and seed.
outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(outer.split(manifest, groups=manifest["speaker_id"]))

train_df = manifest.iloc[train_idx].reset_index(drop=True)
test_df = manifest.iloc[test_idx].reset_index(drop=True)

assert set(train_df.speaker_id).isdisjoint(set(test_df.speaker_id))
assert set(train_df.original_file).isdisjoint(set(test_df.original_file))

print("Training speakers:", train_df.speaker_id.nunique())
print("Final test speakers:", test_df.speaker_id.nunique())
print("Training WAVs:", len(train_df))
print("Final test WAVs:", len(test_df))
print("Final test original recordings:", int((~test_df.is_augmented).sum()))

Training speakers: 42
Final test speakers: 11
Training WAVs: 3522
Final test WAVs: 918
Final test original recordings: 153


In [3]:
# Extract RAW MFCCs. Do not normalize here.
SR = 22050
N_MFCC = 20
N_FRAMES = 40

def extract_mfcc(path):
    signal, _ = librosa.load(path, sr=SR, mono=True)
    mfcc = librosa.feature.mfcc(y=signal, sr=SR, n_mfcc=N_MFCC)
    if mfcc.shape[1] < N_FRAMES:
        mfcc = np.pad(mfcc, ((0, 0), (0, N_FRAMES - mfcc.shape[1])))
    else:
        mfcc = mfcc[:, :N_FRAMES]
    return mfcc.astype(np.float32)

X_train_raw = np.stack([extract_mfcc(p) for p in train_df.path])
X_test_raw = np.stack([extract_mfcc(p) for p in test_df.path])

encoder = LabelEncoder()
y_train = encoder.fit_transform(train_df.label)
y_test = encoder.transform(test_df.label)
classes = encoder.classes_.astype(str)

# Keep original_file identifiers so Notebook 03 can weight each source recording equally.
train_original_ids = train_df.original_file.to_numpy()
test_original_ids = test_df.original_file.to_numpy()
train_is_original = (~train_df.is_augmented).to_numpy()
test_is_original = (~test_df.is_augmented).to_numpy()

print("Raw training MFCC:", X_train_raw.shape)
print("Raw final-test MFCC:", X_test_raw.shape)
print("Labels:", list(classes))

Raw training MFCC: (3522, 20, 40)
Raw final-test MFCC: (918, 20, 40)
Labels: [np.str_('anu'), np.str_('close browser'), np.str_('close calculator'), np.str_('help'), np.str_('mute'), np.str_('next'), np.str_('open browser'), np.str_('open calculator'), np.str_('pause'), np.str_('play'), np.str_('previous'), np.str_('screenshot'), np.str_('sleep'), np.str_('unmute')]


In [4]:
# Create 10 stratified speaker-grouped folds within the outer training partition.
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_id = np.full(len(train_df), -1, dtype=np.int32)

for fold, (_, val_idx) in enumerate(
    cv.split(X_train_raw, y_train, groups=train_df.speaker_id), start=1
):
    fold_id[val_idx] = fold

assert (fold_id > 0).all()
assert set(np.unique(fold_id)) == set(range(1, N_FOLDS + 1))

summary_rows = []
for fold in range(1, N_FOLDS + 1):
    val_mask = fold_id == fold
    fit_mask = ~val_mask
    fit_speakers = set(train_df.loc[fit_mask, "speaker_id"])
    val_speakers = set(train_df.loc[val_mask, "speaker_id"])
    assert fit_speakers.isdisjoint(val_speakers)

    val_original_mask = val_mask & train_is_original
    summary_rows.append({
        "fold": fold,
        "fit_speakers": len(fit_speakers),
        "validation_speakers": len(val_speakers),
        "fit_wavs": int(fit_mask.sum()),
        "validation_wavs_all": int(val_mask.sum()),
        "validation_original_recordings": int(val_original_mask.sum())
    })

fold_summary = pd.DataFrame(summary_rows)
display(fold_summary)

,fold,fit_speakers,validation_speakers,fit_wavs,validation_wavs_all,validation_original_recordings
0,1,38,4,3174,348,58
1,2,37,5,3108,414,69
2,3,38,4,3192,330,55
3,4,38,4,3192,330,55
4,5,38,4,3180,342,57
5,6,38,4,3192,330,55
6,7,38,4,3180,342,57
7,8,37,5,3108,414,69
8,9,38,4,3186,336,56
9,10,38,4,3186,336,56


In [5]:
# Save raw arrays, metadata, and folds.
np.savez_compressed(
    OUT / "speaker_split_mfcc_10fold_raw.npz",
    X_train_raw=X_train_raw,
    X_test_raw=X_test_raw,
    y_train=y_train,
    y_test=y_test,
    train_original_ids=train_original_ids,
    test_original_ids=test_original_ids,
    train_is_original=train_is_original,
    test_is_original=test_is_original,
    fold_id=fold_id,
    classes=classes
)

joblib.dump(encoder, OUT / "label_encoder_10fold.joblib")
train_df.assign(cv_fold=fold_id).to_csv(OUT / "train_manifest_10fold.csv", index=False)
test_df.to_csv(OUT / "test_manifest_10fold.csv", index=False)
fold_summary.to_csv(OUT / "10fold_partition_summary.csv", index=False)

print("Saved raw MFCCs and fold assignments:", OUT.resolve())

Saved raw MFCCs and fold assignments: C:\Users\ADMIN\Desktop\Sukriti\VIT\SL\AnuAssistant\Anu_Revision_Notebooks\anu_revision_output
